In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

from src.DataLoad_ResaleTransactions import load_transactions
from src.InterimAnalysis_ResaleTransactions import (market_snapshot,
                                                    yearly_summary,
                                                    monthly_summary,
                                                    town_summary,
                                                    flat_type_summary,
                                                    add_bands
                                                    )
from src.InterimPlot_ResaleTransactions import (output_path,
                                                _save,
                                                plot_yearly_volume_price,
                                                plot_monthly_volume_price,
                                                plot_town_ranking,
                                                plot_price_distribution,
                                                plot_band_vs_price
                                                )

sns.set_theme(style = 'whitegrid')
pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
df = load_transactions()
df.shape

(239265, 17)

In [3]:
# yearly trend
yearly = yearly_summary(df)
yearly

,trx_year,transactions,mean_price,median_price,mean_pps,median_pps,mean_area,median_area
0,2017,20337,443743.44,410000.00,4577.22,4278.85,97.74,96.00
1,2018,21552,441310.44,408000.00,4509.78,4208.79,98.45,97.00
2,2019,22168,432180.07,400000.00,4476.56,4175.22,97.10,94.00
3,2020,23313,452312.74,425000.00,4669.74,4354.84,97.63,93.00
4,2021,29057,511417.39,483000.00,5248.62,4915.25,98.25,93.00
5,2022,26702,549701.07,525000.00,5723.56,5373.13,96.89,93.00
6,2023,25740,571814.76,550000.00,6071.10,5708.33,95.12,93.00
7,2024,27817,612597.98,590000.00,6493.21,6097.56,95.35,93.00
8,2025,25072,652509.23,628000.00,6953.13,6500.00,94.90,93.00
9,2026,17507,661168.73,630000.00,6999.89,6483.05,95.61,93.00


In [4]:
# yearly trend plot
plot_yearly_volume_price(yearly)

Saved: E:\AI study\HDB_resale_market_analysis\src\..\figures\yearly_volume_price.png


'E:\\AI study\\HDB_resale_market_analysis\\src\\..\\figures\\yearly_volume_price.png'

In [5]:
# monthly trend
monthly = monthly_summary(df)
monthly

,trx_year,trx_month,transactions,mean_price,median_price,mean_pps,median_pps,mean_area,median_area
0,2017,1,1176,427378.21,403000.00,4520.88,4293.80,95.71,95.00
1,2017,2,1080,447295.95,415000.00,4583.13,4316.06,98.47,96.00
2,2017,3,1889,444852.25,415000.00,4620.33,4328.36,97.16,95.00
3,2017,4,1821,438553.12,406600.00,4563.92,4298.51,97.06,95.00
4,2017,5,1961,443663.26,410000.00,4590.68,4269.84,97.42,96.00
...,...,...,...,...,...,...,...,...,...
112,2026,5,2127,661140.83,630000.00,7028.19,6506.85,95.15,93.00
113,2026,6,2123,663715.46,625000.00,7066.80,6513.27,95.06,93.00
114,2026,7,2651,660655.18,630000.00,6947.41,6407.41,96.39,93.00
115,2026,8,2519,667258.02,635000.00,6979.35,6466.67,96.68,93.00


In [6]:
# monthly trend plot
plot_monthly_volume_price(monthly)

Saved: E:\AI study\HDB_resale_market_analysis\src\..\figures\monthly_volume_price.png


'E:\\AI study\\HDB_resale_market_analysis\\src\\..\\figures\\monthly_volume_price.png'

In [7]:
# price distribution plot
plot_price_distribution(df)

Saved: E:\AI study\HDB_resale_market_analysis\src\..\figures\price_distribution.png


'E:\\AI study\\HDB_resale_market_analysis\\src\\..\\figures\\price_distribution.png'

In [8]:
# price summary (pct)
price_stats = df['resale_price'].describe(percentiles = [0.05, 0.25, 0.5, 0.75, 0.95])
pps_stats = df['price_per_sqm'].describe(percentiles = [0.05, 0.25, 0.5, 0.75, 0.95])
print('=== Resale Price ===')
display(price_stats)
print('\n=== Price per sqm ===')
display(pps_stats)

=== Resale Price ===


count    239265.00
mean     534525.25
std      192112.86
min      140000.00
5%       280000.00
25%      390000.00
50%      503000.00
75%      640000.00
95%      900000.00
max     1728000.00
Name: resale_price, dtype: float64


=== Price per sqm ===


count   239265.00
mean      5590.00
std       1676.84
min       2089.55
5%        3495.15
25%       4405.59
50%       5310.00
75%       6347.83
95%       8817.20
max      16148.94
Name: price_per_sqm, dtype: float64

In [9]:
# outliers view (pct = 0.5%)
q_low = df['resale_price'].quantile(0.005)
q_high = df['resale_price'].quantile(0.995)

print(f'Bottom 0.5% threshold: {q_low:.2f}')
print(f'Top 0.5% threshold: {q_high:.2f}')

high_outliers = df[df['resale_price'] >= q_high].sort_values('resale_price', ascending = False)
low_outliers = df[df['resale_price'] <= q_low].sort_values('resale_price', ascending = True)

print(f'\nHigh outliers: {len(high_outliers)}')
display(high_outliers[['trx_date', 'remaining_lease_month', 'town', 'flat_type', 'floor_area_sqm', 'resale_price', 'price_per_sqm']].head(10))

print(f'\nLow outliers: {len(low_outliers)}')
display(low_outliers[['trx_date', 'remaining_lease_month', 'town', 'flat_type', 'floor_area_sqm', 'resale_price', 'price_per_sqm']].head(10))

Bottom 0.5% threshold: 220000.00
Top 0.5% threshold: 1200000.00

High outliers: 1267


,trx_date,remaining_lease_month,town,flat_type,floor_area_sqm,resale_price,price_per_sqm
225133,2026-04-01,1105,BUKIT MERAH,5 ROOM,113.00,1728000.00,15292.04
232562,2026-02-01,1071,QUEENSTOWN,5 ROOM,122.00,1700000.00,13934.43
225104,2026-08-01,1062,BUKIT MERAH,5 ROOM,112.00,1688888.00,15079.36
211772,2025-06-01,1079,QUEENSTOWN,5 ROOM,122.00,1658888.00,13597.44
223528,2026-08-01,721,BISHAN,EXECUTIVE,162.00,1650000.00,10185.19
232566,2026-06-01,1067,QUEENSTOWN,5 ROOM,122.00,1650000.00,13524.59
225101,2026-03-01,1066,BUKIT MERAH,5 ROOM,112.00,1648888.00,14722.21
199180,2025-11-01,1018,BISHAN,5 ROOM,120.00,1632000.00,13600.00
225927,2026-05-01,1004,CENTRAL AREA,5 ROOM,105.00,1630000.00,15523.81
223506,2026-08-01,1008,BISHAN,5 ROOM,120.00,1620000.00,13500.00



Low outliers: 1390


,trx_date,remaining_lease_month,town,flat_type,floor_area_sqm,resale_price,price_per_sqm
67360,2020-02-01,599,TOA PAYOH,3 ROOM,67.00,140000.00,2089.55
166901,2023-12-01,1139,WOODLANDS,2 ROOM,47.00,150000.00,3191.49
65604,2020-01-01,749,TOA PAYOH,2 ROOM,43.00,150000.00,3488.37
58761,2019-10-01,579,GEYLANG,2 ROOM,45.00,150000.00,3333.33
163619,2023-08-01,1095,TAMPINES,3 ROOM,67.00,150000.00,2238.81
66217,2020-02-01,649,BUKIT MERAH,1 ROOM,31.00,157000.00,5064.52
73560,2020-07-01,571,GEYLANG,2 ROOM,42.00,160000.00,3809.52
52901,2019-07-01,583,GEYLANG,2 ROOM,42.00,160000.00,3809.52
35232,2018-09-01,616,GEYLANG,2 ROOM,47.00,160000.00,3404.26
67905,2020-03-01,648,BUKIT MERAH,1 ROOM,31.00,160000.00,5161.29


In [10]:
# year-on-year change
yoy = yearly.copy(deep = True)
yoy['trx_yoy_pct'] = yoy['transactions'].pct_change() * 100
yoy['price_yoy_pct'] = yoy['median_price'].pct_change() * 100
yoy['pps_yoy_pct'] = yoy['median_pps'].pct_change() * 100
yoy[['trx_year', 'transactions', 'trx_yoy_pct', 'median_price', 'price_yoy_pct', 'median_pps', 'pps_yoy_pct']]

,trx_year,transactions,trx_yoy_pct,median_price,price_yoy_pct,median_pps,pps_yoy_pct
0,2017,20337,NaN,410000.00,NaN,4278.85,NaN
1,2018,21552,5.97,408000.00,-0.49,4208.79,-1.64
2,2019,22168,2.86,400000.00,-1.96,4175.22,-0.80
3,2020,23313,5.17,425000.00,6.25,4354.84,4.30
4,2021,29057,24.64,483000.00,13.65,4915.25,12.87
5,2022,26702,-8.10,525000.00,8.70,5373.13,9.32
6,2023,25740,-3.60,550000.00,4.76,5708.33,6.24
7,2024,27817,8.07,590000.00,7.27,6097.56,6.82
8,2025,25072,-9.87,628000.00,6.44,6500.00,6.60
9,2026,17507,-30.17,630000.00,0.32,6483.05,-0.26


In [11]:
# month-on-month change
mom = monthly.copy(deep = True)
mom['trx_mom_pct'] = mom['transactions'].pct_change() * 100
mom['price_mom_pct'] = mom['median_price'].pct_change() * 100
mom['pps_mom_pct'] = mom['median_pps'].pct_change() * 100
mom[['trx_year', 'trx_month', 'transactions', 'trx_mom_pct', 'median_price', 'price_mom_pct', 'median_pps', 'pps_mom_pct']]

,trx_year,trx_month,transactions,trx_mom_pct,median_price,price_mom_pct,median_pps,pps_mom_pct
0,2017,1,1176,NaN,403000.00,NaN,4293.80,NaN
1,2017,2,1080,-8.16,415000.00,2.98,4316.06,0.52
2,2017,3,1889,74.91,415000.00,0.00,4328.36,0.28
3,2017,4,1821,-3.60,406600.00,-2.02,4298.51,-0.69
4,2017,5,1961,7.69,410000.00,0.84,4269.84,-0.67
...,...,...,...,...,...,...,...,...
112,2026,5,2127,10.04,630000.00,0.00,6506.85,0.27
113,2026,6,2123,-0.19,625000.00,-0.79,6513.27,0.10
114,2026,7,2651,24.87,630000.00,0.80,6407.41,-1.63
115,2026,8,2519,-4.98,635000.00,0.79,6466.67,0.92
